# LFP anatomy check — ripples test shank orientation, theta tests MEC vs LEC

Loads the stage outputs of `run_lfp_anatomy_check.py` (run under sbatch; see `LFP_ANATOMY_CHECK.md`) and plots.
Nothing heavy runs here: `RUN_MODE = 'load'`.

In [1]:
RUN_MODE = 'load'
import json, pickle, sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, '.')
import lfp_anatomy_check as lac

OUT = lac.OUT_DIR
FIG = lac.FIG_DIR; FIG.mkdir(parents=True, exist_ok=True)
load = lambda p: pickle.load(open(p, 'rb'))
anat_tab = pd.read_csv(OUT / 'contact_anatomy.csv')
anat = {m: lac.Anatomy(d) for m, d in anat_tab.groupby('mouse')}
for g in sorted(OUT.glob('*gates.json')):
    print(g.name, json.loads(g.read_text()).get('pass'))

anatomy_gates.json True
probemap_ah08_gates.json True
probemap_ah10_gates.json True
probemap_ly05_gates.json True
probemap_ly06_gates.json True


probemap_ly07_gates.json True
raw_ly06_20250618_20250619_s4_gates.json True
raw_ly06_20250618_20250619_s4_medpsd_gates.json True
raw_ly06_20250618_20250619_sb0_gates.json True
raw_ly06_20250618_20250619_sb0_medpsd_gates.json True
raw_ly07_20250620_20250623_s4_gates.json True
raw_ly07_20250620_20250623_s4_medpsd_gates.json True


raw_ly07_20250620_20250623_sb5_gates.json True
raw_ly07_20250620_20250623_sb5_medpsd_gates.json True
speed_gates.json True
units_all_gates.json True


## Gates

In [2]:
for g in sorted(OUT.glob('*gates.json')):
    d = json.loads(g.read_text())
    print(f'--- {g.name}'); print({k: v for k, v in d.items() if not isinstance(v, (list, dict)) or len(str(v)) < 300})

--- anatomy_gates.json
{'ah08': {'current_vs_htsv': 1.0, 'mirror_position_dev_vox_max': 0.0, 'mirror_label_vs_twin_lookup': 1.0}, 'ah10': {'current_vs_htsv': 1.0, 'mirror_position_dev_vox_max': 0.0, 'mirror_label_vs_twin_lookup': 1.0}, 'ly05': {'current_vs_htsv': 1.0, 'mirror_position_dev_vox_max': 0.0, 'mirror_label_vs_twin_lookup': 1.0}, 'ly06': {'current_vs_htsv': 1.0, 'mirror_position_dev_vox_max': 0.0, 'mirror_label_vs_twin_lookup': 1.0}, 'ly07': {'current_vs_htsv': 1.0, 'mirror_position_dev_vox_max': 0.0, 'mirror_label_vs_twin_lookup': 1.0}, 'channel_regions_agree': 1.0, 'channel_regions_n': 1901, 'pass': True}
--- probemap_ah08_gates.json
{'a0_all_pass': True, 'a0_configs_failed': [], 'a0_median_r_own': 0.9982674091581016, 'pass': True}
--- probemap_ah10_gates.json
{'a0_all_pass': True, 'a0_configs_failed': [], 'a0_median_r_own': 0.9981002551892119, 'pass': True}
--- probemap_ly05_gates.json
{'a0_all_pass': True, 'a0_configs_failed': [], 'a0_median_r_own': 0.9983484484173188, 'p

## Probe map: atlas labels next to physiology, per mouse

Left two panels: atlas groups under the current shank convention and under the mirror. Then ripple gain
(reference-triggered, circle = reference channel), rest-state ripple-band excess, theta-state theta excess,
and 300-450 Hz power (cell layers, brain surface). Shank 0 is drawn on the left.

In [3]:
pm = load(OUT / 'probemap.pkl')
ch = pm['channels'].copy()
ch['log_mua'] = np.log10(ch['rest_mua_power'])
for m in sorted(ch.mouse.unique()):
    d = ch[ch.mouse == m]
    lac.plot_probe_map(d, anat[m], m, ['ripple_gain', 'rest_ripple_excess_db', 'th_theta_excess_db', 'log_mua'],
                       titles=['ripple gain', 'ripple excess (dB)', 'theta excess (dB)', 'log MUA power'],
                       out_path=FIG / f'probemap_{m}.png')
    plt.show()

## Orientation: AUC of ripple measures for ripple structures vs other grey matter

In [4]:
o = pm['orientation']
display(o[['mouse', 'measure', 'auc_current', 'auc_mirror', 'rank_current', 'rank_reversed',
           'n_pos_current', 'n_pos_mirror', 'verdict']])
lac.plot_orientation(o, 'ripple_gain', out_path=FIG / 'probemap_orientation_ripple_gain.png'); plt.show()
display(pm['theta'])
# POST-HOC (added after the ly06 probe map; reported beside, never instead of, the verdicts above):
# discordant contacts only, ranks within bank; AUC > 0.5 favours current
display(pm.get('discordant'))
obc = pm.get('orientation_by_config')
if obc is not None:
    display(obc[obc.measure == 'ripple_gain'][['mouse', 'config', 'auc_current', 'auc_mirror', 'n_pos_current', 'n_pos_mirror', 'verdict']])

,mouse,measure,auc_current,auc_mirror,rank_current,rank_reversed,n_pos_current,n_pos_mirror,verdict
0,ah08,ripple_gain,0.704250,0.677074,11,13,338,310,inconclusive
1,ah08,rest_ripple_excess_db,0.771826,0.690201,3,21,338,310,inconclusive
2,ah08,ripple_rate_hz,0.889538,0.858457,4,19,338,310,inconclusive
3,ah10,ripple_gain,0.228190,0.164938,4,24,425,437,inconclusive
4,ah10,rest_ripple_excess_db,0.587293,0.478703,3,23,425,437,inconclusive
5,ah10,ripple_rate_hz,0.785983,0.794010,16,11,425,437,inconclusive
6,ly05,ripple_gain,0.787835,0.398202,1,24,287,254,current
7,ly05,rest_ripple_excess_db,0.906406,0.557705,1,21,287,254,current
8,ly05,ripple_rate_hz,0.653182,0.560038,5,24,287,254,inconclusive
9,ly06,ripple_gain,0.570190,0.472400,6,15,471,453,inconclusive


,mouse,derivation,measure,auc_entm_entl_current,auc_l5_current,n_entm_current,n_entl_current,n_entm5_current,n_entl5_current,auc_entm_entl_mirror,auc_l5_mirror,n_entm_mirror,n_entl_mirror,n_entm5_mirror,n_entl5_mirror
0,ah08,monopolar,th_theta_excess_db,NaN,NaN,0,192,0,66,NaN,NaN,0,192,0,65
1,ah08,monopolar,th_rel_theta,NaN,NaN,0,192,0,66,NaN,NaN,0,192,0,65
2,ah08,bipolar,th_theta_excess_db,NaN,NaN,0,176,0,58,NaN,NaN,0,176,0,57
3,ah10,monopolar,th_theta_excess_db,NaN,NaN,0,117,0,46,0.132759,NaN,5,116,0,46
4,ah10,monopolar,th_rel_theta,NaN,NaN,0,117,0,46,0.139655,NaN,5,116,0,46
5,ah10,bipolar,th_theta_excess_db,NaN,NaN,0,107,0,40,0.274691,NaN,3,108,0,40
6,ly05,monopolar,th_theta_excess_db,0.092308,0.000000,13,130,12,63,0.059001,0.002924,18,129,18,57
7,ly05,monopolar,th_rel_theta,0.086982,0.000000,13,130,12,63,0.059001,0.001949,18,129,18,57
8,ly05,bipolar,th_theta_excess_db,0.509091,0.761171,11,120,11,59,0.791492,0.774764,16,119,16,53
9,ly06,monopolar,th_theta_excess_db,0.723380,0.311702,48,109,20,48,0.151600,0.000000,48,113,18,46


,mouse,measure,auc_discordant,n_current_only,n_mirror_only,p_favours_current,p_favours_mirror
0,ah08,ripple_gain,1.000000,23,19,0.000999,1.000000
1,ah08,rest_ripple_excess_db,1.000000,23,19,0.000999,1.000000
2,ah08,ripple_rate_hz,1.000000,23,19,0.000999,1.000000
3,ah10,ripple_gain,0.771786,79,69,0.000999,1.000000
4,ah10,rest_ripple_excess_db,0.879630,79,69,0.000999,1.000000
5,ah10,ripple_rate_hz,0.314815,79,69,1.000000,0.000999
6,ly05,ripple_gain,0.955685,263,225,0.000999,1.000000
7,ly05,rest_ripple_excess_db,0.887603,263,225,0.000999,1.000000
8,ly05,ripple_rate_hz,0.732716,263,225,0.000999,1.000000
9,ly06,ripple_gain,0.723751,111,90,0.000999,1.000000


,mouse,config,auc_current,auc_mirror,n_pos_current,n_pos_mirror,verdict
0,ah08,config_1,NaN,NaN,0,0,inconclusive
3,ah08,config_2,0.984604,0.699667,57,60,current
6,ah08,config_3,NaN,0.000000,173,143,inconclusive
9,ah08,config_4,0.150515,0.991364,97,100,mirror
12,ah08,config_5,NaN,NaN,0,0,inconclusive
15,ah10,config_1,0.837749,0.657062,60,59,inconclusive
18,ah10,config_2,0.799708,0.169984,171,169,current
21,ah10,config_3,0.924848,0.457643,165,181,current
24,ah10,config_4,0.137931,0.145714,29,28,inconclusive
27,ah10,config_5,NaN,NaN,0,0,inconclusive


## Raw pilot (ly06, ly07): both columns of the recorded bank

In [5]:
raws = {p.stem[4:]: load(p) for p in sorted(OUT.glob('raw_*.pkl'))}
for tag, r in raws.items():
    print(tag, r['gates'])
    m = tag.split('_')[0]
    c = r['channels'].copy(); c['log_mua'] = np.log10(c['rest_mua_power'])
    lac.plot_probe_map(c, anat[m], tag, ['ripple_gain', 'rest_ripple_excess_db', 'th_theta_excess_db', 'log_mua'],
                       titles=['ripple gain', 'ripple excess (dB)', 'theta excess (dB)', 'log MUA power'],
                       y_max=720, out_path=FIG / f'raw_{tag}.png')
    plt.show()
if raws:
    ro = pd.concat([r['orientation'].assign(recording=t) for t, r in raws.items()], ignore_index=True)
    display(ro[['recording', 'measure', 'auc_current', 'auc_mirror', 'rank_current', 'verdict']])
    lac.plot_orientation(ro, 'ripple_gain', out_path=FIG / 'raw_orientation_ripple_gain.png'); plt.show()
    display(pd.concat([r['theta'].assign(recording=t) for t, r in raws.items()], ignore_index=True))
    rd = OUT / 'raw_discordant.csv'
    if rd.exists():
        display(pd.read_csv(rd))   # POST-HOC discordant test per recording

ly06_20250618_20250619_s4 {'recday': 'ly06_20250618_20250619', 'session': 's4', 'kind': 'task', 'folder': '2025-06-19_12-28-25', 'geometry_vs_sort_max_um': 0.0, 'decimation_r_median': 0.9999998974719246, 'decimation_r_min': 0.002081464691055919, 'clock_lag_s': 0.0, 'clock_r': 0.4423640473846968, 'theta_speed_lag_s': -5.0, 'theta_speed_rho': 0.03732489821816241, 'theta_speed_rho_at_0': -0.14438243346656984, 'round_trip_max_abs_diff': 0, 'pass': True}


/nfs/nhome/live/aharris/.conda/envs/maze_ephys/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1424: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(


ly06_20250618_20250619_s4_medpsd {'recday': 'ly06_20250618_20250619', 'session': 's4', 'kind': 'task', 'folder': '2025-06-19_12-28-25', 'geometry_vs_sort_max_um': 0.0, 'decimation_r_median': 0.9999998974719246, 'decimation_r_min': 0.002081464691055919, 'clock_lag_s': 0.0, 'clock_r': 0.4423640473846968, 'theta_speed_lag_s': -5.0, 'theta_speed_rho': 0.03732489821816241, 'theta_speed_rho_at_0': -0.14438243346656984, 'round_trip_max_abs_diff': 0, 'pass': True}


/nfs/nhome/live/aharris/.conda/envs/maze_ephys/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1424: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(


ly06_20250618_20250619_sb0 {'recday': 'ly06_20250618_20250619', 'session': 'sb0', 'kind': 'sleep', 'folder': '2025-06-18_11-31-16', 'geometry_vs_sort_max_um': 0.0, 'decimation_r_median': 0.9999998999308557, 'decimation_r_min': 0.0020820398778324573, 'round_trip_max_abs_diff': 0, 'pass': True}


...

ly07_20250620_20250623_sb5 {'recday': 'ly07_20250620_20250623', 'session': 'sb5', 'kind': 'sleep', 'folder': '2025-06-23_11-09-47', 'geometry_vs_sort_max_um': 0.0, 'decimation_r_median': 0.9999999099589086, 'decimation_r_min': 0.0004010492925874069, 'round_trip_max_abs_diff': 0, 'pass': True}


ly07_20250620_20250623_sb5_medpsd {'recday': 'ly07_20250620_20250623', 'session': 'sb5', 'kind': 'sleep', 'folder': '2025-06-23_11-09-47', 'geometry_vs_sort_max_um': 0.0, 'decimation_r_median': 0.9999998880292293, 'decimation_r_min': -0.003417222946807497, 'round_trip_max_abs_diff': 0, 'pass': True}


,recording,measure,auc_current,auc_mirror,rank_current,verdict
0,ly06_20250618_20250619_s4,ripple_gain,NaN,NaN,-1,inconclusive
1,ly06_20250618_20250619_s4,rest_ripple_excess_db,0.999312,0.453244,1,current
2,ly06_20250618_20250619_s4,ripple_rate_hz,0.207772,0.728349,23,mirror
3,ly06_20250618_20250619_s4_medpsd,ripple_gain,NaN,NaN,-1,inconclusive
4,ly06_20250618_20250619_s4_medpsd,rest_ripple_excess_db,0.999140,0.465386,1,current
5,ly06_20250618_20250619_s4_medpsd,ripple_rate_hz,0.207772,0.728349,23,mirror
6,ly06_20250618_20250619_sb0,ripple_gain,0.996903,0.546589,1,current
7,ly06_20250618_20250619_sb0,rest_ripple_excess_db,1.000000,0.546310,1,current
8,ly06_20250618_20250619_sb0,ripple_rate_hz,0.677947,0.574937,15,inconclusive
9,ly06_20250618_20250619_sb0_medpsd,ripple_gain,0.995928,0.536898,1,current


,derivation,measure,auc_entm_entl_current,auc_l5_current,n_entm_current,n_entl_current,n_entm5_current,n_entl5_current,auc_entm_entl_mirror,auc_l5_mirror,n_entm_mirror,n_entl_mirror,n_entm5_mirror,n_entl5_mirror,recording
0,monopolar,th_theta_excess_db,0.652479,1.000000,96,222,38,94,0.053060,0.000000,96,222,38,94,ly06_20250618_20250619_s4
1,monopolar,th_rel_theta,0.491092,0.973684,96,222,38,94,0.194570,0.010926,96,222,38,94,ly06_20250618_20250619_s4
2,bipolar,th_theta_excess_db,0.810773,0.874300,88,206,34,86,0.480791,0.402980,88,206,34,86,ly06_20250618_20250619_s4
3,monopolar,th_theta_excess_db,0.514942,0.990379,96,222,38,94,0.194713,0.000288,96,222,38,94,ly06_20250618_20250619_s4_medpsd
4,monopolar,th_rel_theta,0.488264,0.987550,96,222,38,94,0.200810,0.003163,96,222,38,94,ly06_20250618_20250619_s4_medpsd
5,bipolar,th_theta_excess_db,0.817458,0.873599,88,206,34,86,0.473552,0.364099,88,206,34,86,ly06_20250618_20250619_s4_medpsd
6,monopolar,th_theta_excess_db,0.326357,0.000000,96,222,38,94,0.225387,0.000000,96,222,38,94,ly06_20250618_20250619_sb0
7,monopolar,th_rel_theta,0.354921,0.000000,96,222,38,94,0.184901,0.000000,96,222,38,94,ly06_20250618_20250619_sb0
8,bipolar,th_theta_excess_db,0.818850,0.869748,88,206,34,86,0.363657,0.331395,88,206,34,86,ly06_20250618_20250619_sb0
9,monopolar,th_theta_excess_db,0.682362,0.076684,96,222,38,94,0.197285,0.000000,96,222,38,94,ly06_20250618_20250619_sb0_medpsd


,recording,measure,auc_discordant,n_current_only,n_mirror_only,p_favours_current,p_favours_mirror
0,ly06_20250618_20250619_s4,ripple_gain,NaN,0,0,NaN,NaN
1,ly06_20250618_20250619_s4,rest_ripple_excess_db,1.000000,55,57,0.000999,1.000000
2,ly06_20250618_20250619_s4,ripple_rate_hz,0.107974,55,57,1.000000,0.000999
3,ly06_20250618_20250619_s4_medpsd,ripple_gain,NaN,0,0,NaN,NaN
4,ly06_20250618_20250619_s4_medpsd,rest_ripple_excess_db,1.000000,55,57,0.000999,1.000000
5,ly06_20250618_20250619_s4_medpsd,ripple_rate_hz,0.107974,55,57,1.000000,0.000999
6,ly06_20250618_20250619_sb0,ripple_gain,1.000000,55,57,0.000999,1.000000
7,ly06_20250618_20250619_sb0,rest_ripple_excess_db,1.000000,55,57,0.000999,1.000000
8,ly06_20250618_20250619_sb0,ripple_rate_hz,0.568262,55,57,0.099900,0.901099
9,ly06_20250618_20250619_sb0_medpsd,ripple_gain,1.000000,55,57,0.000999,1.000000


## Theta against signed distance to the ENTl/ENTm border (EC contacts; − ENTl, + ENTm)

In [6]:
for tag, r in raws.items():
    if 'task' in r['gates'].get('kind', '') or r['gates'].get('kind') == 'task':
        b = r['bipolar'].assign(mouse=tag.split('_')[0])
        for mapping in ('current', 'mirror'):
            lac.plot_border_profile(b, 'th_theta_excess_db', anat, mapping, ylabel=f'bipolar theta excess (dB)\n{tag}, {mapping}',
                                    out_path=FIG / f'border_theta_{tag}_{mapping}.png'); plt.show()

/ceph/behrens/adam_harris/Taskspace_abstraction_lEC/code/lfp_anatomy_check.py:1333: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axes = plt.subplots(1, len(mice), figsize=(1.7 * len(mice) + 0.4, 2.0), sharey=True, squeeze=False)


## Units: ACG theta excess during running (every unit-recording, spike-only)

Faint dots are unit-recordings (the same neuron recurs across the 5 blocks); large markers are per-mouse medians.

In [7]:
ut = load(OUT / 'units_theta.pkl')
u = ut['units']
for mapping in ('current', 'mirror'):
    lac.plot_unit_groups(u, 'theta_excess_db', mapping, ylabel='ACG theta excess (dB)',
                         out_path=FIG / f'units_theta_excess_{mapping}.png'); plt.show()
display(ut['fractions'])
display(ut['contrasts'])

,mouse,group_current,n,frac_rhythmic,median_excess_db,median_ratio,median_burst_frac
0,ah08,ENTl-deep,152,0.309211,4.784570,0.713112,0.034478
1,ah08,ENTl-sup,472,0.211864,3.722512,0.829829,0.026606
2,ah10,ENTl-deep,331,0.510574,8.084239,1.910970,0.022985
3,ah10,ENTl-sup,9,0.333333,5.868090,0.673134,0.031250
4,ah10,ENTm,22,0.227273,2.892768,0.421636,0.024243
5,ah10,HPF,236,0.792373,16.159990,5.971169,0.060365
6,ah10,other,61,0.770492,13.525534,5.357591,0.043456
7,ly05,ENTl-deep,127,0.299213,3.684179,0.824368,0.026469
8,ly05,ENTl-sup,40,0.125000,2.765723,0.635493,0.024663
9,ly05,ENTm,33,0.424242,6.903119,0.981648,0.029906


,mouse,contrast,value,n_a,n_b,diff,p_shuffle,mapping
0,ah10,ENTm - ENTl-deep,theta_excess_db,22,331,-5.191471,0.991009,current
1,ly05,ENTm - ENTl-deep,theta_excess_db,33,127,3.218940,0.045954,current
2,ly06,ENTm - ENTl-deep,theta_excess_db,48,112,-7.869027,1.000000,current
3,ly07,ENTm - ENTl-deep,theta_excess_db,156,87,-0.983530,0.743257,current
4,ah10,ENTm - ENTl-deep,theta_ratio,22,331,-1.489334,1.000000,current
...,...,...,...,...,...,...,...,...
79,ly07,HPF - ENTl-deep,theta_ratio,122,174,-0.226062,0.902098,mirror
80,ah10,HPF - ENTl-deep,burst_frac,229,364,-0.011139,1.000000,mirror
81,ly05,HPF - ENTl-deep,burst_frac,70,213,-0.006318,0.922078,mirror
82,ly06,HPF - ENTl-deep,burst_frac,8,142,-0.021259,0.944056,mirror


## Raw-pilot units: ripple modulation and theta locking by label group, current vs mirror

In [8]:
if raws:
    ru = pd.concat([r['units'] for r in raws.values()], ignore_index=True)
    sleep = ru[ru.session.str.startswith('sb')]
    task = ru[~ru.session.str.startswith('sb')]
    for mapping in ('current', 'mirror'):
        lac.plot_unit_groups(sleep, 'ripple_ratio', mapping, ylabel='ripple modulation (rate ratio)', log=True,
                             out_path=FIG / f'raw_units_ripple_{mapping}.png'); plt.show()
        lac.plot_unit_groups(task, 'mrl_ref', mapping, ylabel='theta MRL (probe-median theta)',
                             out_path=FIG / f'raw_units_mrl_{mapping}.png'); plt.show()